Importing necessary libraries

In [13]:
from typing import Tuple, List, Dict

import pandas as pd
import matplotlib as plt
import folium
import sqlite3
from IPython.display import display # unnevcessary?

Extracting stops and their location from the Izmir tram database

In [14]:
pd.set_option('display.precision', 2)

def extract_stops_from_db(db_path: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    query = "SELECT stop_name, stop_lat, stop_lon FROM stops"
    df = pd.read_sql_query(query, conn)
    conn.close()
    df.columns = ['site', 'latitude', 'longitude']
    return df

The dataframe from the tutorial

In [15]:
df_tutorial = pd.DataFrame(
    [['hotel',              48.8527, 2.3542],
     ['Sacre Coeur',        48.8867, 2.3431],
     ['Louvre',             48.8607, 2.3376],
     ['Montmartre',         48.8872, 2.3388],
     ['Port de Suffren',    48.8577, 2.2902],
     ['Arc de Triomphe',    48.8739, 2.2950],
     ['Av. Champs Élysées', 48.8710, 2.3036],
     ['Notre Dame',         48.8531, 2.3498],
     ['Tour Eiffel',        48.8585, 2.2945]],
    columns=pd.Index(['site', 'latitude', 'longitude'], name='paris')
)

Visualise the extracted data

In [16]:
df_sites

,site,latitude,longitude
0,Aliağa,38.79,26.97
1,Biçerova,38.75,26.96
2,Hatundere,38.69,27.02
3,Menemen,38.60,27.08
4,Egekent 2,38.56,27.04
5,Ulukent,38.55,27.04
6,Egekent,38.51,27.05
7,Atasanayi,38.50,27.05
8,Çiğli,38.49,27.06
9,Mavişehir,38.48,27.08


In [28]:
df_sites = extract_stops_from_db('izmir_izban_static_data.db')

print()
df_sites_1 = df_sites.loc[0:19]
# select in a specific order (allow duplicates) using the site index
df_sites_2 = df_sites.set_index('site').loc[['Hilal', 'Alsancak', 'Hilal']].reset_index()

df_sites_tail = df_sites.loc[20:].copy()
df_sites_3 = df_sites_tail[~df_sites_tail['site'].isin(['Hilal', 'Alsancak'])].reset_index(drop=True)
df_sites_fixed = pd.concat([df_sites_1, df_sites_2], ignore_index=True)
df_sites_fixed = pd.concat([df_sites_fixed, df_sites_3], ignore_index=True)
df_sites_fixed

,site,latitude,longitude
0,Aliağa,38.79,26.97
1,Biçerova,38.75,26.96
2,Hatundere,38.69,27.02
3,Menemen,38.60,27.08
4,Egekent 2,38.56,27.04
5,Ulukent,38.55,27.04
6,Egekent,38.51,27.05
7,Atasanayi,38.50,27.05
8,Çiğli,38.49,27.06
9,Mavişehir,38.48,27.08


In [ ]:
avg_location = df_sites[['latitude', 'longitude']].mean() # to display the average location of all stops
map_izmir = folium.Map(location=avg_location, zoom_start=13, tiles="cartodb positron") 


for site in df_sites.itertuples():
    marker = folium.Marker(location=(site.latitude, site.longitude), tooltip=site.site)
    marker.add_to(map_izmir)

map_izmir

c:\Users\mertv\Documents\Projects\VS Code projects\Project_Reliable_NMBS\Project-reliable-NMBS\.venv\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])


display route


In [30]:
df_route = df_sites_fixed.copy()
df_route.index.name = 'visit_order'

df_route

,site,latitude,longitude
visit_order,,,
0,Aliağa,38.79,26.97
1,Biçerova,38.75,26.96
2,Hatundere,38.69,27.02
3,Menemen,38.60,27.08
4,Egekent 2,38.56,27.04
5,Ulukent,38.55,27.04
6,Egekent,38.51,27.05
7,Atasanayi,38.50,27.05
8,Çiğli,38.49,27.06


connect stops

In [31]:
df_route_segments = df_route.join(
    df_route.shift(-1),  # map each stop to its next stop
    rsuffix='_next'
).dropna()  # last stop has no "next one", so drop it

visualise connection


In [32]:
map_izmir = folium.Map(location=avg_location, zoom_start=13,tiles="cartodb positron")

for stop in df_route_segments.itertuples():
    # marker for current stop
    marker = folium.Marker(location=(stop.latitude, stop.longitude),
                           tooltip=stop.site)
    # line for the route segment connecting current to next stop
    line = folium.PolyLine(
        locations=[(stop.latitude, stop.longitude), 
                   (stop.latitude_next, stop.longitude_next)],
        tooltip=f"{stop.site} to {stop.site_next}",
    )
    # add elements to the map
    marker.add_to(map_izmir)
    line.add_to(map_izmir)

# maker for last stop wasn't added in for loop, so adding it now 
folium.Marker(location=(stop.latitude_next, stop.longitude_next),
              tooltip=stop.site_next).add_to(map_izmir);

map_izmir

c:\Users\mertv\Documents\Projects\VS Code projects\Project_Reliable_NMBS\Project-reliable-NMBS\.venv\Lib\site-packages\folium\utilities.py:95: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coords = (location[0], location[1])
